In [ ]:
import sqlite3
from typing import Literal

import pandas as pd
import pendulum
import ollama
from tqdm import tqdm
from loguru import logger

from unga81.config import DATABASE_PATH
from unga81.database import get_all_countries, get_db_connection

In [ ]:
conn = get_db_connection()
df = get_all_countries(conn=conn)
logger.info(f"{df.shape[0]} speeches in the database")

In [ ]:
df.info()

In [ ]:
summary_prompt = """Summarize the following speech. Ignore the speaker and focus on the content. Output ONLY the summary, nothing else.

Speech:
'''{text}'''
"""

countries_mentioned_prompt = """Extract all countries mentioned in this text in exactly what order they appear. 
Format as a comma-separated list with country names only (e.g., "France, Japan, Germany").
Ignore proper nouns that are not standalone countries (cities like Paris don't count).
Output ONLY the ordered list, nothing else.

Text:
'''{text}'''
"""

risks_prompt = """Summarize the risks and challenges discussed in the provided speech.
Identify the points that require attention. Use bullet points to present a concise list of the main risks/concerns.

For each risk, briefly state it AND its source/cause. Do not invent risks beyond what is implied or stated.
Output ONLY the the bullet points, nothing else.

Text:
'''{text}'''
"""

haiku_prompt = """Craft a 3-line haiku inspired by the key themes or messages of the speech. Reply only with the lines of the haiku.
Output ONLY the haiku, nothing else.

Speech:
'''{text}'''
"""

single_word_prompt = """Extract a single word that best represents this entire presidential speech's core message or main point. 
The word should be one the speaker themselves might use when asking "What is THIS about?"
Output ONLY the single word, nothing else.

Speech:
'''{text}'''
"""

hashtags_prompt = """Create a set of relevant hashtags representing this presidential speech's content and themes. 

Requirements:
- Include 5-7 hashtags total
- Prioritize topics that would appear on Twitter/X when discussing this speech
- Mix broad topic tags with more specific keywords 
- Use #hashtags only (no spaces)

Output ONLY the hashtag list, nothing else.

Example output:
#PresidentialSpeech #ClimatePolicy #InternationalRelations #GlobalCooperation #NewAdministration #EconomicGrowth #ForeignPolicyMatters #VotersFirst

Speech:
'''{text}'''
"""

headlines_prompt = """Generate a single impactful newspaper headline for each one of the following styles: 
[leftist, alt-right, environmental, financial, sports] from the following speech. 
Each headline should be unique and accurately reflect the content of the speech. Do not include the name of the paper.

The output should be as follows. Output ONLY the headlines. Nothing else:
- [leftist] <leftist headline>
- [alt-right] <alt-right headline>
- [environmental] <environmental headline>
- [financial] <financial headline>
- [sports] <sports headline>

Speech: 
'''{text}'''"""

yoda_prompt = """Craft a short advice in Yoda's voice, based on the content of the speech. Don't describe it, just write what Yoda would say. 
Output ONLY the Yoda advice, nothing else.

Speech:
'''{text}'''
"""

prompts = [
    ("summary", summary_prompt),
    ("countries_mentioned", countries_mentioned_prompt),
    ("risks", risks_prompt),
    ("haiku", haiku_prompt),
    ("single_word", single_word_prompt),
    ("hashtags", hashtags_prompt),
    ("headlines", headlines_prompt),
    ("yoda", yoda_prompt),
]

In [ ]:
conn = sqlite3.connect(DATABASE_PATH)

In [ ]:
def insert_into_analysis(conn: sqlite3.Connection, data: dict):
    # TODO update with yoda text
    query_delete = """delete from analysis where country = ?;"""
    query_insert = """INSERT INTO analysis
                    (country,
                     iso_3,
                     summary,
                     countries_mentioned,
                     risks,
                     haiku,
                     single_word,
                     hashtags,
                     headlines,
                     yoda)
                    VALUES(?, ?, ?, ?, ?, ?, ?, ?, ?, ?);"""
    with conn:
        cursor = conn.cursor()
        cursor.execute(query_delete, (data["country"],))
        conn.commit()

        cursor.execute(
            query_insert,
            (
                data["country"],
                data["iso_3"],
                data["summary"],
                data["countries_mentioned"],
                data["risks"],
                data["haiku"],
                data["single_word"],
                data["hashtags"],
                data["headlines"],
                data["yoda"],
            ),
        )
        conn.commit()


PromptType = Literal[
    "summary",
    "countries_mentioned",
    "risks",
    "haiku",
    "single_word",
    "hashtags",
    "headlines",
    "yoda",
]


def update_analysis_column(
    conn: sqlite3.Connection,
    country: str,
    iso_3: str,
    column_name: PromptType,
    text: str,
):
    """Update a specific column in the `analysis` table.

    Posible values in `column_name` are the ones in `PromptType`:
    - summary
    - countries_mentioned
    - risks
    - haiku
    - single_word
    - hashtags
    - headlines
    - yoda

    Args:
        conn (sqlite3.Connection): Database connection.
        country (str): Country to be updated in full name.
        iso_3 (str): Code ISO 3 of the country to be updated.
        column_name (PromptType): Column name in the analysis table.
        text (str): Value of the column to be inserted.
    """
    query_check = """select * from analysis where country = ?;"""
    query_update = f"""update analysis set {column_name} = ? where country = ?;"""
    with conn:
        cursor = conn.cursor()
        cursor.execute(query_check, (country,))
        rows = cursor.fetchone()
        if rows is None:
            # insert
            logger.info(f"Inserting data into 'analysis.{column_name}' for {country}")
            cursor.execute(
                f"insert into analysis (country, iso_3, {column_name}) values (?, ?, ?);",
                (
                    country,
                    iso_3,
                    text,
                ),
            )
            conn.commit()

        else:
            # update
            logger.info(f"Updating data into 'analysis.{column_name}' for {country}")
            cursor.execute(
                query_update,
                (
                    text,
                    country,
                ),
            )
            conn.commit()


def check_analysis_column(
    conn: sqlite3.Connection, country: str, column_name: PromptType
) -> bool:
    """Check if a record about the `country` exists in the database to decide
    whether to insert a new record or to update the existing one.

    Args:
        conn (sqlite3.Connection): Database connection.
        country (str): Country to check in the analysis table.
        column_name (PromptType): _description_

    Returns:
        bool: _description_
    """
    query = f"""select {column_name} from analysis where country = ?;"""
    with conn:
        cursor = conn.cursor()
        cursor.execute(query, (country,))
        row = cursor.fetchone()
        cursor.close()

        if row is None:
            return False
        else:
            if row[0] is None:
                return False
            else:
                return True

In [ ]:
def run_model_prompt(model: str, prompt: str, text: str, think: bool = False):
    result = ollama.generate(
        model=model,
        prompt=prompt.format(text=text),
        think=think,
    )
    return result


# TODO implement update function by country and prompt and check if extsts
def update_prompt_result(
    prompt_name: PromptType,
    prompt: str,
    model: str,
    country: str,
    iso_3: str,
    text: str,
    overwrite: bool = False,
):

    if (
        check_analysis_column(conn=conn, country=country, column_name=prompt_name)
        and not overwrite
    ):
        return

    logger.info(f"Running '{prompt_name}' prompt with model '{model}' for {country}")
    result = run_model_prompt(model=model, prompt=prompt, text=text)
    # result = ollama.generate(
    #     model=model,
    #     prompt=prompt.format(text=text),
    #     think=False,
    # )
    logger.info(f"Updating '{prompt_name}' with '{model}' for {country}")
    update_analysis_column(
        conn=conn,
        country=country,
        iso_3=iso_3,
        column_name=prompt_name,
        text=str(result["response"]),
    )

In [ ]:
for i, r in df.iterrows():
    start = pendulum.now()
    logger.info(f"{i + 1}/{df.shape[0]} processing {r['country']}")
    update_prompt_result(
        prompt_name="summary",
        prompt=summary_prompt,
        model="qwen2.5:3b",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        prompt_name="countries_mentioned",
        prompt=countries_mentioned_prompt,
        model="llama3.2",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        prompt_name="risks",
        prompt=risks_prompt,
        model="gemma3:4b",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        prompt_name="haiku",
        prompt=haiku_prompt,
        model="qwen3.5:0.8b",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        prompt_name="single_word",
        prompt=single_word_prompt,
        model="llama3.2",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        prompt_name="hashtags",
        prompt=hashtags_prompt,
        model="llama3.2",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        prompt_name="headlines",
        prompt=headlines_prompt,
        model="qwen2.5:3b",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        prompt_name="yoda",
        prompt=yoda_prompt,
        model="llama3.2",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )

    end = pendulum.now()
    logger.info(
        f"{i + 1}/{df.shape[0]} processed {r['country']} in {(end - start).in_words()}"
    )

In [ ]:
import textwrap

# test different models
models = [
    "llama3.2",
    "gemma3:4b",
    "qwen2.5:3b",
    "qwen3.5:0.8b",
    "mistral:7b",
]
# summary                   qwen2.5:3b
# countries_mentioned       llama3.2
# risks                     gemma3:4b, mistral:7b
# haiku                     qwen3.5:0.8b
# single_word               llama3.2, qwen2.5:3b
# hashtags                  llama3.2
# headlines                 qwen2.5:3b, mistral:7b
# yoda                      llama3.2, mistral:7b

# prompt_name, prompt, country = "summary", summary_prompt, "Chile"
# prompt_name, prompt, country = "summary", summary_prompt, "Brazil"
# prompt_name, prompt, country = "summary", summary_prompt, "Argentina"
# prompt_name, prompt, country = "haiku", haiku_prompt, "Chile"
prompt_name, prompt, country = "haiku", haiku_prompt, "Brazil"
# prompt_name, prompt, country = "haiku", haiku_prompt, "Argentina"
# prompt_name, prompt, country = "risks", risks_prompt, "Chile"                               # gemma3:4b, mistral:7b
# prompt_name, prompt, country = "countries_mentioned", countries_mentioned_prompt, "Chile"   # llama3.2
# prompt_name, prompt, country = "single_word", single_word_prompt, "Chile"  # llama3.2, qwen2.5:3b
# prompt_name, prompt, country = (
#     "single_word",
#     single_word_prompt,
#     "United States of America",
# )  # llama3.2, qwen2.5:3b
# prompt_name, prompt, country = "headlines", headlines_prompt, "Chile"  # qwen2.5:3b, mistral:7b
# prompt_name, prompt, country = "hashtags", hashtags_prompt, "Brazil"  # llama3.2
# prompt_name, prompt, country = "yoda", yoda_prompt, "Chile"                                 # llama3.2, mistral:7b
text = df[df["country"] == country]["full_speech"].values[0]

with open("evidence.md", "a+", encoding="utf-8") as f:
    f.write(f"# `{prompt_name}` on '{country}'\n\n")
    total_start = pendulum.now()
    for model in models:
        start = pendulum.now()
        print(f"{prompt_name} {model}")
        f.write(f"## `{prompt_name}` - `{model}`\n\n")
        result = run_model_prompt(model=model, prompt=prompt, text=text)
        # print(textwrap.fill(result["response"], 80))
        print(result["response"])
        f.write(f'{result["response"]}\n\n')
        end = pendulum.now()
        print(f"{(end - start).in_words()}")
        f.write(f"{(end - start).in_words()}\n\n")
        print("=" * 20)
    total_end = pendulum.now()
    print(
        f"Evaluation finished in {(total_end - total_start).in_words()} for {len(models)} models"
    )
    f.write(
        f"{(total_end - total_start).in_words()} total for {len(models)} models\n\n"
    )
    f.write("---\n\n")

In [ ]:
llm = "llama3.2"
for i, r in df.iterrows():
    data = {}
    data["country"] = r["country"]
    data["iso_3"] = r["iso_3"]
    pbar = tqdm(prompts)
    for name, prompt in pbar:
        pbar.set_description(f"{i+1}/{df.shape[0]} {r["country"]} - {name}")
        if check_analysis_column(conn=conn, country=data["country"], column_name=name):
            continue
        result = ollama.generate(
            model=llm,
            prompt=prompt.format(text=r["full_speech"]),
            think=False,
        )
        data[name] = str(result["response"])
        update_analysis_column(
            conn=conn,
            country=data["country"],
            iso_3=data["iso_3"],
            column_name=name,
            text=str(result["response"]),
        )
    # insert_into_analysis(conn=conn, data=data)

In [ ]:
llm = "llama3.2"
prompt = """Write a succinct conclusion about the following speech. 
The conclusion will be included in a general report to the machines and robots council that eliminated all humans.

Speech:
'''{text}'''"""
prompt = """Infer the political leaning of the speaker of the following speech.
Output ONLY the political leaning, nothing else.

Speech:
'''{text}'''"""
prompt = """Craft a short advice in Yoda's voice, based on the content of the speech. Don't describe it, just write what Yoda would say. 
Output ONLY the Yoda advice, nothing else.

Speech:
'''{text}'''"""
result = ollama.generate(
    model=llm,
    prompt=prompt.format(text=df[df["country"] == "Chile"].full_speech.values[0]),
    think=False,
)
print(result["response"])